# NLP Assignment PS-5 (Group-164)
## Intelligent News Article Classification and Fake News Detection
**BITS Pilani — AIMLCZG530**

---
## System Architecture & Design

The following diagrams describe the full NLP pipeline architecture. 
Every component shown is implemented in the cells below.


In [ ]:
import pathlib
for candidate in [
    pathlib.Path("system-component-diag.txt"),
    pathlib.Path("bits-nlp-assignment-main/system-component-diag.txt"),
]:
    if candidate.exists():
        print(candidate.read_text()); break
else:
    print("system-component-diag.txt not found — place it next to this notebook.")


In [ ]:
import pathlib
for candidate in [
    pathlib.Path("more-detailed-arch.txt"),
    pathlib.Path("bits-nlp-assignment-main/more-detailed-arch.txt"),
]:
    if candidate.exists():
        print(candidate.read_text()); break
else:
    print("more-detailed-arch.txt not found — place it next to this notebook.")


In [ ]:
import pathlib
for candidate in [
    pathlib.Path("final_design.txt"),
    pathlib.Path("bits-nlp-assignment-main/final_design.txt"),
]:
    if candidate.exists():
        print(candidate.read_text()); break
else:
    print("final_design.txt not found — place it next to this notebook.")


---


In [ ]:
import sys
!{sys.executable} -m pip install -q \
    nltk gensim scikit-learn numpy scipy pandas \
    matplotlib seaborn torch spacy hmmlearn tqdm
!{sys.executable} -m spacy download en_core_web_sm -q
print("All packages installed.")


In [ ]:
import pathlib, subprocess, sys

zip_path = pathlib.Path('./fake-news-detection-datasets.zip')
if not zip_path.exists():
    print('Downloading dataset...')
    r = subprocess.run(
        ['curl', '-L', '-o', str(zip_path),
         'https://www.kaggle.com/api/v1/datasets/download/emineyetm/fake-news-detection-datasets'],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        raise RuntimeError(f'Download failed: {r.stderr}')
    print('Download complete.')
else:
    print(f'Dataset zip already present: {zip_path}')


In [ ]:
import re, ssl, zipfile, string, pathlib, urllib.request, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from collections import Counter
from IPython.display import display, HTML
from tqdm.auto import tqdm
import nltk
import spacy
from spacy import displacy
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag, ne_chunk
from nltk.chunk import RegexpParser
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, roc_curve, auc,
    precision_recall_curve, average_precision_score
)
from sklearn.decomposition import PCA
from sklearn.preprocessing import label_binarize
from hmmlearn import hmm as hmmlib
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')

# Fix SSL for NLTK downloads (macOS)
try:
    ssl._create_default_https_context = ssl._create_unverified_context
except AttributeError:
    pass

for pkg in ['punkt','punkt_tab','stopwords','wordnet',
            'averaged_perceptron_tagger','averaged_perceptron_tagger_eng',
            'maxent_ne_chunker','maxent_ne_chunker_tab','words']:
    nltk.download(pkg, quiet=True)

nlp_spacy = spacy.load('en_core_web_sm')
lemmatizer = WordNetLemmatizer()
STOPWORDS   = set(stopwords.words('english'))
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
print(f'All imports OK  |  device = {DEVICE}')


In [ ]:
extract_dir = pathlib.Path('./fake-news-detection-datasets')
if not extract_dir.exists():
    with zipfile.ZipFile('./fake-news-detection-datasets.zip') as z:
        z.extractall('.')

csv_files = sorted(pathlib.Path('.').rglob('*.csv'))
print('Found CSVs:', [str(p) for p in csv_files])

def _load_csv(p: pathlib.Path) -> pd.DataFrame:
    df = pd.read_csv(p)
    text_col  = next((c for c in df.columns if 'text'  in c.lower()), df.select_dtypes('object').columns[0])
    title_col = next((c for c in df.columns if 'title' in c.lower()), None)
    content   = (df[title_col].fillna('') + '. ') if title_col else ''
    df['full_content'] = content + df[text_col].fillna('')
    df['is_fake'] = 1 if 'fake' in p.stem.lower() else 0
    return df[['full_content', 'is_fake']]

target_files = [p for p in csv_files
                if any(k in p.stem.lower() for k in ('fake','true','real'))]
if not target_files:
    raise FileNotFoundError('No fake/true CSVs found — check extraction.')

full_df = (pd.concat([_load_csv(p) for p in target_files], ignore_index=True)
             .dropna(subset=['full_content'])
             .sample(frac=1, random_state=RANDOM_SEED)
             .reset_index(drop=True))

print(f'\nTotal articles : {len(full_df):,}')
print(full_df['is_fake'].value_counts().rename({0:'Real',1:'Fake'}).to_string())


## Component 1 — Text Preprocessor

Produces three token flavours from raw article text:
- **spacy_raw_string** — unchanged, for SpaCy POS/parsing
- **sequential_tokens** — lowercased, lemmatised, punctuation-stripped (for Neural LM & HMM)
- **semantic_tokens** — sequential minus stopwords (for GloVe)


In [ ]:
POLITICAL_TERMS = {
    'president','senate','congress','democrat','republican','government',
    'election','vote','ballot','policy','legislation','minister','parliament',
    'white house','administration','federal','campaign','candidate','political',
    'opposition','senator','governor','mayor','diplomat','treaty','sanction',
}

def master_preprocessor(raw_text: str) -> dict:
    raw_text = str(raw_text).strip()
    lowered  = raw_text.lower()
    tokens   = word_tokenize(lowered)
    # filter punctuation, numbers, single chars, and whitespace-only
    cleaned  = [
        lemmatizer.lemmatize(w) for w in tokens
        if w not in string.punctuation
        and not w.isnumeric()
        and len(w) > 1
        and w.strip()
    ]
    semantic    = [w for w in cleaned if w not in STOPWORDS]
    pol_terms   = [w for w in semantic if w in POLITICAL_TERMS]
    return {
        'spacy_raw_string':  raw_text,
        'sequential_tokens': cleaned,
        'semantic_tokens':   semantic,
        'political_terms':   pol_terms,
    }

# Smoke test
_t = master_preprocessor('President Biden signed new federal legislation in Congress today.')
print('sequential :', _t['sequential_tokens'])
print('semantic   :', _t['semantic_tokens'])
print('political  :', _t['political_terms'])


In [ ]:
train_df, test_df = train_test_split(
    full_df, test_size=0.2, random_state=RANDOM_SEED, stratify=full_df['is_fake']
)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

def tokenise_df(df: pd.DataFrame, desc: str) -> pd.DataFrame:
    results = [master_preprocessor(t) for t in tqdm(df['full_content'], desc=desc)]
    df = df.copy()
    df['tokens_seq']   = [r['sequential_tokens'] for r in results]
    df['tokens_sem']   = [r['semantic_tokens']   for r in results]
    df['pol_terms']    = [r['political_terms']   for r in results]
    return df

train_df = tokenise_df(train_df, 'Tokenising train')
test_df  = tokenise_df(test_df,  'Tokenising test')
print(f'Train: {len(train_df):,}  |  Test: {len(test_df):,}')


---
## Task 1 — Neural Language Model

A Feedforward Neural LM predicts the next word given a context window of 3 tokens.
Perplexity scores are computed per article and used as features for fake news detection.


In [ ]:
def build_vocab(df_col, max_size: int = 10_000) -> tuple:
    counts = Counter(
        tok for tokens in df_col
        if isinstance(tokens, list)
        for tok in tokens
    )
    vocab     = {'<UNK>': 0}
    idx2word  = {0: '<UNK>'}
    for i, (w, _) in enumerate(counts.most_common(max_size - 1), start=1):
        vocab[w]    = i
        idx2word[i] = w
    return vocab, idx2word, len(vocab)

word_to_idx, idx_to_word, vocab_size = build_vocab(train_df['tokens_seq'])
print(f'Vocabulary size: {vocab_size:,}')


In [ ]:
CONTEXT_SIZE = 3

def generate_windows(df_col, w2i: dict, ctx: int = 3) -> tuple:
    X, Y = [], []
    for tokens in df_col:
        if not isinstance(tokens, list) or len(tokens) <= ctx:
            continue
        ids = [w2i.get(t, 0) for t in tokens]
        for i in range(len(ids) - ctx):
            X.append(ids[i:i+ctx])
            Y.append(ids[i+ctx])
    return torch.tensor(X, dtype=torch.long), torch.tensor(Y, dtype=torch.long)

print('Building context windows...')
X_lm, Y_lm = generate_windows(train_df['tokens_seq'], word_to_idx, CONTEXT_SIZE)

# 80/20 train-val split for the LM
n_val = int(len(X_lm) * 0.1)
X_lm_train, X_lm_val = X_lm[n_val:], X_lm[:n_val]
Y_lm_train, Y_lm_val = Y_lm[n_val:], Y_lm[:n_val]
print(f'LM windows — train:{X_lm_train.shape}  val:{X_lm_val.shape}')


In [ ]:
class FeedforwardNeuralLM(nn.Module):
    """3-layer FFNN language model: Embedding → Linear+ReLU+Dropout → Linear."""
    def __init__(self, vocab_size: int, embed_dim: int = 100,
                 ctx: int = 3, hidden: int = 256):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.fc1     = nn.Linear(ctx * embed_dim, hidden)
        self.relu    = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2     = nn.Linear(hidden, vocab_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        e = self.embeddings(x).view(x.size(0), -1)
        return self.fc2(self.dropout(self.relu(self.fc1(e))))

lm_model     = FeedforwardNeuralLM(vocab_size, ctx=CONTEXT_SIZE).to(DEVICE)
criterion_lm = nn.CrossEntropyLoss()
optimizer_lm = optim.Adam(lm_model.parameters(), lr=1e-3)
print(lm_model)


In [ ]:
train_loader = DataLoader(TensorDataset(X_lm_train, Y_lm_train),
                          batch_size=512, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_lm_val, Y_lm_val),
                          batch_size=512, shuffle=False)

train_losses, val_losses = [], []
NUM_EPOCHS = 5

for epoch in range(NUM_EPOCHS):
    # ── training ──
    lm_model.train()
    t_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer_lm.zero_grad()
        loss = criterion_lm(lm_model(xb), yb)
        loss.backward()
        optimizer_lm.step()
        t_loss += loss.item()
    t_loss /= len(train_loader)

    # ── validation ──
    lm_model.eval()
    v_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            v_loss += criterion_lm(lm_model(xb), yb).item()
    v_loss /= len(val_loader)

    train_losses.append(t_loss)
    val_losses.append(v_loss)
    print(f'Epoch {epoch+1}/{NUM_EPOCHS}  '
          f'train_loss={t_loss:.4f}  val_loss={v_loss:.4f}  '
          f'val_perplexity={np.exp(v_loss):.2f}')

# Save model weights
torch.save(lm_model.state_dict(), 'lm_model.pt')
print('\nModel saved to lm_model.pt')


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, NUM_EPOCHS+1), train_losses, 'o-', label='Train Loss')
ax.plot(range(1, NUM_EPOCHS+1), val_losses,   's--', label='Validation Loss')
ax.set_xlabel('Epoch'); ax.set_ylabel('Cross-Entropy Loss')
ax.set_title('Neural Language Model — Training & Validation Loss')
ax.legend(); plt.tight_layout(); plt.show()


In [ ]:
def article_perplexity(tokens: list, model: nn.Module,
                        w2i: dict, ctx: int = 3) -> float:
    """Compute per-article perplexity using trained Neural LM."""
    if not isinstance(tokens, list) or len(tokens) <= ctx:
        return 1000.0
    ids = [w2i.get(t, 0) for t in tokens]
    X   = torch.tensor([ids[i:i+ctx] for i in range(len(ids)-ctx)],
                       dtype=torch.long).to(DEVICE)
    Y   = torch.tensor(ids[ctx:], dtype=torch.long).to(DEVICE)
    crit = nn.CrossEntropyLoss(reduction='none')
    model.eval()
    with torch.no_grad():
        losses = crit(model(X), Y)
    return float(np.exp(losses.mean().item()))

print('Scoring perplexity — train...')
train_df['perplexity'] = [
    article_perplexity(t, lm_model, word_to_idx)
    for t in tqdm(train_df['tokens_seq'], desc='Train perplexity')
]
print('Scoring perplexity — test...')
test_df['perplexity'] = [
    article_perplexity(t, lm_model, word_to_idx)
    for t in tqdm(test_df['tokens_seq'], desc='Test perplexity')
]
print('\nPerplexity stats by label (train):')
print(train_df.groupby('is_fake')['perplexity']
      .describe().rename(index={0:'Real',1:'Fake'}))


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
cap = 3000
for label, name, color in [(0,'Real','steelblue'),(1,'Fake','tomato')]:
    vals = train_df[train_df['is_fake']==label]['perplexity'].clip(upper=cap)
    ax.hist(vals, bins=60, alpha=0.6, label=name, color=color, density=True)
ax.set_xlabel('Perplexity (capped at 3000)')
ax.set_ylabel('Density')
ax.set_title('Perplexity Distribution — Fake vs Real Articles')
ax.legend(); plt.tight_layout(); plt.show()


In [ ]:
def predict_next_words(prompt: str, model: nn.Module, w2i: dict,
                       i2w: dict, ctx: int = 3, top_k: int = 5) -> list:
    """Given a text prompt, predict the top-k most likely next words."""
    proc    = master_preprocessor(prompt)
    tokens  = proc['sequential_tokens'][-ctx:]
    if len(tokens) < ctx:
        tokens = ['<UNK>'] * (ctx - len(tokens)) + tokens
    ids = torch.tensor([[w2i.get(t, 0) for t in tokens]],
                       dtype=torch.long).to(DEVICE)
    model.eval()
    with torch.no_grad():
        logits = model(ids)[0]
    probs   = torch.softmax(logits, dim=0)
    top_idx = probs.argsort(descending=True)[:top_k]
    return [(i2w.get(int(i), '<UNK>'), round(float(probs[i]), 4)) for i in top_idx]

print('Next-Word Prediction Demo\n' + '='*40)
for prompt in [
    'The president signed a new',
    'Scientists have discovered a major',
    'The election results were',
]:
    preds = predict_next_words(prompt, lm_model, word_to_idx, idx_to_word)
    print(f'\nPrompt : "{prompt}"')
    print(f'Top-5  : {preds}')


---
## Task 2 — Vector Embeddings (GloVe)

News articles are converted to 100-dimensional dense vectors by mean-pooling GloVe token vectors.
These document vectors capture semantic meaning and feed directly into the downstream classifiers.


In [ ]:
glove_dir  = pathlib.Path('./glove')
glove_file = glove_dir / 'glove.6B.100d.txt'

if not glove_file.exists():
    glove_dir.mkdir(exist_ok=True)
    print('Downloading GloVe 6B 100d (~330 MB)...')
    glove_zip = glove_dir / 'glove.6B.zip'
    urllib.request.urlretrieve(
        'https://nlp.stanford.edu/data/glove.6B.zip', glove_zip,
        reporthook=lambda b,bs,t: print(
            f'  {min(b*bs,t)/1e6:.0f}/{t/1e6:.0f} MB', end='\r')
    )
    with zipfile.ZipFile(glove_zip) as z:
        z.extract('glove.6B.100d.txt', glove_dir)
    glove_zip.unlink()
    print('\nGloVe ready.')
else:
    print(f'GloVe already present: {glove_file}')


In [ ]:
EMBED_DIM = 100
print('Loading GloVe vectors...')
glove: dict = {}
with open(glove_file, encoding='utf-8') as f:
    for line in f:
        parts = line.split()
        glove[parts[0]] = np.array(parts[1:], dtype=np.float32)
if not glove:
    raise RuntimeError('GloVe loaded 0 vectors — delete glove/ and re-run.')
print(f'Loaded {len(glove):,} vectors  dim={EMBED_DIM}')


In [ ]:
def tokens_to_glove_vec(tokens: list, glove: dict,
                         dim: int = 100) -> np.ndarray:
    """Mean-pool GloVe vectors for in-vocabulary tokens."""
    vecs = [glove[t] for t in tokens if t in glove]
    return (np.mean(vecs, axis=0).astype(np.float32)
            if vecs else np.zeros(dim, dtype=np.float32))

print('Building GloVe document vectors...')
train_df['glove_vec'] = [
    tokens_to_glove_vec(t, glove)
    for t in tqdm(train_df['tokens_sem'], desc='Train GloVe')
]
test_df['glove_vec'] = [
    tokens_to_glove_vec(t, glove)
    for t in tqdm(test_df['tokens_sem'], desc='Test GloVe')
]

X_glove_train = np.vstack(train_df['glove_vec'])
X_glove_test  = np.vstack(test_df['glove_vec'])

# OOV rate
oov = train_df['tokens_sem'].apply(
    lambda ts: sum(1 for t in ts if t not in glove) / max(len(ts),1)
).mean()
print(f'\nTrain GloVe matrix : {X_glove_train.shape}')
print(f'Test  GloVe matrix : {X_glove_test.shape}')
print(f'Avg OOV rate       : {oov:.1%}')


In [ ]:
# Pre-build full normalised matrix for cosine lookups (all 400k words)
print('Building cosine lookup index...')
_nn_words  = list(glove.keys())
_nn_mat    = np.vstack([glove[w] for w in _nn_words])
_nn_norms  = np.linalg.norm(_nn_mat, axis=1, keepdims=True)
_nn_normed = _nn_mat / np.where(_nn_norms == 0, 1, _nn_norms)
print(f'Index built: {len(_nn_words):,} words')

def nearest_neighbors(word: str, top_k: int = 5) -> list:
    if word not in glove:
        return [f"'{word}' not in GloVe vocab"]
    q    = glove[word] / (np.linalg.norm(glove[word]) + 1e-9)
    sims = _nn_normed @ q
    idx  = sims.argsort()[::-1][1:top_k+1]
    return [(_nn_words[i], round(float(sims[i]), 4)) for i in idx]

print('\nGloVe Nearest Neighbours:')
for w in ['fake','news','president','election','crime','vaccine']:
    print(f'  {w:12s} → {nearest_neighbors(w)}')


In [ ]:
def article_similarity(text_a: str, text_b: str) -> float:
    """Cosine similarity between two article GloVe vectors."""
    va = tokens_to_glove_vec(master_preprocessor(text_a)['semantic_tokens'], glove)
    vb = tokens_to_glove_vec(master_preprocessor(text_b)['semantic_tokens'], glove)
    denom = (np.linalg.norm(va) * np.linalg.norm(vb)) + 1e-9
    return float(np.dot(va, vb) / denom)

a1 = test_df['full_content'].iloc[0]
a2 = test_df['full_content'].iloc[1]
a3 = test_df['full_content'].iloc[100]
print(f'Similarity (article 0 vs 1)  : {article_similarity(a1, a2):.4f}  (same dataset chunk)')
print(f'Similarity (article 0 vs 100): {article_similarity(a1, a3):.4f}  (distant article)')


In [ ]:
words_to_plot = ['fake','news','president','election','crime',
                 'media','government','war','economy','health']
known  = [w for w in words_to_plot if w in glove]
vecs   = np.vstack([glove[w] for w in known])
coords = PCA(n_components=2, random_state=RANDOM_SEED).fit_transform(vecs)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(coords[:,0], coords[:,1], color='steelblue', s=80, zorder=3)
for i, w in enumerate(known):
    ax.annotate(w, (coords[i,0], coords[i,1]),
                fontsize=11, xytext=(6, 4), textcoords='offset points')
ax.set_title('GloVe Word Vectors — PCA 2D Projection')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
plt.tight_layout(); plt.show()


---
## Task 3 — POS Tagging, NER & Political Terms

SpaCy is used for POS tagging, Named Entity Recognition (NER), keyword extraction, action verb identification, and political term detection.


In [ ]:
def pos_tag_article(text: str) -> dict:
    """Full POS tagging and NER on a single article via SpaCy."""
    doc = nlp_spacy(str(text)[:2000])
    pos_tags = [{'Word': t.text, 'POS': t.pos_, 'Tag': t.tag_,
                 'Dep': t.dep_}
                for t in doc if not t.is_space]
    keywords     = list({t.lemma_.lower() for t in doc
                         if t.pos_ in ('NOUN','PROPN') and not t.is_stop})
    action_verbs = list({t.lemma_ for t in doc if t.pos_ == 'VERB'})
    entities     = [{'Entity': e.text, 'Type': e.label_,
                     'Explanation': spacy.explain(e.label_) or ''}
                    for e in doc.ents]
    pol_terms    = [t.text.lower() for t in doc
                    if t.text.lower() in POLITICAL_TERMS]
    pos_counts   = Counter(t.pos_ for t in doc if not t.is_space)
    return {
        'pos_tags':     pos_tags,
        'keywords':     keywords,
        'action_verbs': action_verbs,
        'entities':     entities,
        'pol_terms':    pol_terms,
        'pos_counts':   dict(pos_counts),
    }

# Demo
sample_text = full_df['full_content'].iloc[0]
pos_res = pos_tag_article(sample_text)
print('=== POS TAGS (first 15 tokens) ===')
display(pd.DataFrame(pos_res['pos_tags'][:15]))
print('\n=== NAMED ENTITIES ===')
display(pd.DataFrame(pos_res['entities'][:10]))
print('\nKeywords    :', pos_res['keywords'][:10])
print('Action verbs:', pos_res['action_verbs'][:8])
print('Political   :', pos_res['pol_terms'][:8])


In [ ]:
# Run POS tagging on full test set (needed for features + analysis)
print('Running POS tagging on test set...')
pos_results_test = [pos_tag_article(t)
                    for t in tqdm(test_df['full_content'], desc='POS tagging')]
test_df['pos_counts']   = [r['pos_counts']   for r in pos_results_test]
test_df['pos_keywords'] = [r['keywords']     for r in pos_results_test]
test_df['pos_entities'] = [r['entities']     for r in pos_results_test]
test_df['pos_pol_terms']= [r['pol_terms']    for r in pos_results_test]
print('Done.')


In [ ]:
# Extract POS-ratio features for the classifier
POS_FEATURE_TAGS = ['ADJ','ADV','VERB','NOUN','PROPN','NUM','INTJ']

def pos_feature_vector(pos_counts: dict) -> np.ndarray:
    """Normalised POS tag ratios as a feature vector."""
    total = sum(pos_counts.values()) or 1
    return np.array([pos_counts.get(tag, 0) / total
                     for tag in POS_FEATURE_TAGS], dtype=np.float32)

print('Extracting POS features for train set...')
pos_results_train = [pos_tag_article(t)
                     for t in tqdm(train_df['full_content'], desc='POS train')]
train_df['pos_counts']    = [r['pos_counts'] for r in pos_results_train]
train_df['pos_pol_terms'] = [r['pol_terms']  for r in pos_results_train]

train_df['pos_feat'] = train_df['pos_counts'].apply(pos_feature_vector)
test_df['pos_feat']  = test_df['pos_counts'].apply(pos_feature_vector)

X_pos_train = np.vstack(train_df['pos_feat'])
X_pos_test  = np.vstack(test_df['pos_feat'])
print(f'POS feature matrix — train:{X_pos_train.shape}  test:{X_pos_test.shape}')


In [ ]:
# POS frequency comparison: Fake vs Real
fake_counts = Counter()
real_counts = Counter()
for _, row in train_df.iterrows():
    target = fake_counts if row['is_fake'] == 1 else real_counts
    total  = sum(row['pos_counts'].values()) or 1
    for tag, cnt in row['pos_counts'].items():
        target[tag] += cnt / total

tags   = ['ADJ','ADV','VERB','NOUN','PROPN','NUM','PUNCT','DET']
n_fake = train_df['is_fake'].sum()
n_real = len(train_df) - n_fake
fake_r = [fake_counts.get(t,0)/n_fake for t in tags]
real_r = [real_counts.get(t,0)/n_real for t in tags]

x = np.arange(len(tags)); w = 0.35
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x - w/2, fake_r, w, label='Fake', color='tomato',    alpha=0.8)
ax.bar(x + w/2, real_r, w, label='Real', color='steelblue', alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(tags)
ax.set_ylabel('Avg normalised frequency')
ax.set_title('POS Tag Distribution — Fake vs Real News')
ax.legend(); plt.tight_layout(); plt.show()


---
## Task 4 — Hidden Markov Model (HMM)

A `CategoricalHMM` is trained on discrete POS tag sequences — one model for fake articles, one for real. Log-likelihood scores and the Viterbi-decoded state sequence serve as features for fake news detection.


In [ ]:
# Universal POS tag set (SpaCy)
POS_TAGS  = ['ADJ','ADP','ADV','AUX','CCONJ','DET','INTJ','NOUN',
             'NUM','PART','PRON','PROPN','PUNCT','SCONJ','SYM','VERB','X']
N_POS     = len(POS_TAGS)
POS2ID    = {p: i for i, p in enumerate(POS_TAGS)}

def article_to_pos_seq(text: str) -> list:
    """Returns a list of POS integer IDs for the article (capped at 500 tokens)."""
    doc = nlp_spacy(str(text)[:1500])
    return [POS2ID.get(t.pos_, N_POS-1)
            for t in doc if not t.is_space]

print('Extracting POS sequences for HMM training...')
train_df['pos_seq'] = [
    article_to_pos_seq(t)
    for t in tqdm(train_df['full_content'], desc='HMM POS seqs')
]
test_df['pos_seq'] = [
    article_to_pos_seq(t)
    for t in tqdm(test_df['full_content'], desc='HMM test seqs')
]
# Drop rows where sequence is too short
train_hmm_df = train_df[train_df['pos_seq'].apply(len) > 10].copy()
print(f'HMM training articles: {len(train_hmm_df):,}')


In [ ]:
def train_categorical_hmm(sequences: list, n_components: int = 6) -> hmmlib.CategoricalHMM:
    """
    Fit a CategoricalHMM on discrete POS-tag sequences.
    CategoricalHMM is the correct model for discrete observations.
    """
    X_concat = np.concatenate(
        [np.array(s, dtype=int).reshape(-1, 1) for s in sequences]
    )
    lengths = [len(s) for s in sequences]
    model   = hmmlib.CategoricalHMM(
        n_components=n_components, n_iter=30,
        random_state=RANDOM_SEED, verbose=False
    )
    model.n_features = N_POS
    model.fit(X_concat, lengths)
    return model

fake_seqs = train_hmm_df[train_hmm_df['is_fake']==1]['pos_seq'].tolist()
real_seqs = train_hmm_df[train_hmm_df['is_fake']==0]['pos_seq'].tolist()

print(f'Training CategoricalHMM on {len(fake_seqs):,} fake articles...')
hmm_fake = train_categorical_hmm(fake_seqs)
print(f'Training CategoricalHMM on {len(real_seqs):,} real articles...')
hmm_real = train_categorical_hmm(real_seqs)
print('HMM training complete.')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, model, title in [
    (axes[0], hmm_fake, 'HMM Transition Matrix — Fake News'),
    (axes[1], hmm_real, 'HMM Transition Matrix — Real News'),
]:
    sns.heatmap(model.transmat_, ax=ax, cmap='Blues',
                xticklabels=[f'S{i}' for i in range(model.n_components)],
                yticklabels=[f'S{i}' for i in range(model.n_components)],
                annot=True, fmt='.2f', linewidths=0.5)
    ax.set_title(title)
    ax.set_xlabel('To State'); ax.set_ylabel('From State')
plt.tight_layout(); plt.show()


In [ ]:
# Viterbi decoding: most likely hidden state sequence
sample_article = test_df['full_content'].iloc[0]
seq = article_to_pos_seq(sample_article)
arr = np.array(seq, dtype=int).reshape(-1, 1)

_, viterbi_states_fake = hmm_fake.decode(arr, algorithm='viterbi')
_, viterbi_states_real = hmm_real.decode(arr, algorithm='viterbi')

# Show first 30 tokens with their POS tags and decoded states
doc_sample = nlp_spacy(sample_article[:500])
tokens_vis = [t.text for t in doc_sample if not t.is_space][:30]
pos_vis    = [t.pos_ for t in doc_sample if not t.is_space][:30]
vit_vis    = viterbi_states_fake[:30]

viterbi_df = pd.DataFrame({
    'Token':       tokens_vis,
    'POS':         pos_vis,
    'HMM State (Fake Model)': vit_vis
})
print('=== Viterbi Decoded State Sequence (first 30 tokens) ===')
display(viterbi_df)


In [ ]:
def hmm_features(text: str) -> np.ndarray:
    """3-dim feature: [ll_fake, ll_real, ll_diff]."""
    seq = article_to_pos_seq(text)
    if len(seq) < 5:
        return np.zeros(3, dtype=np.float32)
    arr    = np.array(seq, dtype=int).reshape(-1, 1)
    ll_f   = hmm_fake.score(arr)
    ll_r   = hmm_real.score(arr)
    return np.array([ll_f, ll_r, ll_f - ll_r], dtype=np.float32)

print('Extracting HMM features...')
train_df['hmm_feats'] = [
    hmm_features(t) for t in tqdm(train_df['full_content'], desc='HMM train')
]
test_df['hmm_feats'] = [
    hmm_features(t) for t in tqdm(test_df['full_content'], desc='HMM test')
]
X_hmm_train = np.vstack(train_df['hmm_feats'])
X_hmm_test  = np.vstack(test_df['hmm_feats'])
print(f'HMM feature matrix — train:{X_hmm_train.shape}  test:{X_hmm_test.shape}')


---
## Task 5 — Syntactic Parsing & SVO Extraction

Dependency parsing extracts the grammatical structure of sentences and identifies Subject-Verb-Object (SVO) triplets that capture factual claims.


In [ ]:
def parse_article(text: str) -> dict:
    """Extract noun phrases, verbs, SVO relationships, and dependency structure."""
    doc = nlp_spacy(str(text)[:2000])
    noun_phrases  = [chunk.text.strip() for chunk in doc.noun_chunks]
    events        = [t.lemma_ for t in doc if t.pos_ == 'VERB']
    relationships = []
    for token in doc:
        if token.pos_ != 'VERB':
            continue
        subj = obj = None
        for child in token.children:
            if child.dep_ in ('nsubj','nsubjpass') and subj is None:
                subj = ' '.join(t.text for t in child.subtree
                                if not t.is_space)
            if child.dep_ in ('dobj','obj','attr') and obj is None:
                obj  = ' '.join(t.text for t in child.subtree
                                if not t.is_space)
        if subj:
            relationships.append({
                'Subject': subj,
                'Verb':    token.lemma_,
                'Object':  obj or '—'
            })
    n_svos = len([r for r in relationships if r['Object'] != '—'])
    return {
        'noun_phrases':   noun_phrases,
        'events':         events,
        'relationships':  relationships,
        'n_noun_phrases': len(noun_phrases),
        'n_svos':         n_svos,
    }

# Demo
parse_res = parse_article(full_df['full_content'].iloc[0])
print('=== NOUN PHRASES ===')
print(parse_res['noun_phrases'][:6])
print('\n=== SVO RELATIONSHIPS ===')
display(pd.DataFrame(parse_res['relationships'][:8]))


In [ ]:
# Dependency tree visualisation using displacy
sample_sent = list(nlp_spacy(full_df['full_content'].iloc[0]).sents)[0]
html = displacy.render(sample_sent, style='dep', jupyter=False,
                       options={'compact': True, 'distance': 90})
display(HTML(f'<div style="overflow-x:auto">{html}</div>'))


In [ ]:
# Parse features for classifier: n_noun_phrases, n_svos (normalised by token count)
def parse_feature_vector(text: str, n_tokens: int) -> np.ndarray:
    res = parse_article(text)
    denom = max(n_tokens, 1)
    return np.array([
        res['n_noun_phrases'] / denom,
        res['n_svos']         / denom,
    ], dtype=np.float32)

print('Extracting parse features for train set...')
train_df['parse_feat'] = [
    parse_feature_vector(row['full_content'], len(row['tokens_seq']))
    for _, row in tqdm(train_df.iterrows(), total=len(train_df), desc='Parse train')
]
print('Extracting parse features for test set...')
test_df['parse_feat'] = [
    parse_feature_vector(row['full_content'], len(row['tokens_seq']))
    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Parse test')
]
X_parse_train = np.vstack(train_df['parse_feat'])
X_parse_test  = np.vstack(test_df['parse_feat'])
print(f'Parse feature matrix — train:{X_parse_train.shape}  test:{X_parse_test.shape}')


---
## Task 6 — Final Classifiers

### Feature Stack
All five tasks feed into a unified feature matrix:

| Source | Features | Dims |
|--------|----------|------|
| GloVe mean-pool | Semantic document vector | 100 |
| HMM log-likelihoods | [ll_fake, ll_real, ll_diff] | 3 |
| Neural LM perplexity | Normalised perplexity score | 1 |
| POS tag ratios | ADJ/ADV/VERB/NOUN/PROPN/NUM/INTJ | 7 |
| Parse ratios | NP density, SVO density | 2 |
| **Total** | | **113** |


In [ ]:
def build_feature_matrix(df: pd.DataFrame) -> np.ndarray:
    """Stack all feature sources into a single matrix."""
    G = np.vstack(df['glove_vec'])                         # (N, 100)
    H = np.vstack(df['hmm_feats'])                          # (N, 3)
    P = (df['perplexity'].clip(upper=5000).values          # (N, 1)
           .reshape(-1,1) / 5000.0)
    S = np.vstack(df['pos_feat'])                           # (N, 7)
    R = np.vstack(df['parse_feat'])                         # (N, 2)
    return np.hstack([G, H, P, S, R]).astype(np.float32)   # (N, 113)

X_train_all = build_feature_matrix(train_df)
X_test_all  = build_feature_matrix(test_df)
y_train     = train_df['is_fake'].values
y_test      = test_df['is_fake'].values
print(f'Feature matrix — train:{X_train_all.shape}  test:{X_test_all.shape}')


### 6A — Binary Classifier: Fake vs Real

In [ ]:
clf_binary = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED, C=1.0)
clf_binary.fit(X_train_all, y_train)
y_pred_bin  = clf_binary.predict(X_test_all)
y_prob_bin  = clf_binary.predict_proba(X_test_all)[:, 1]

print('=== BINARY CLASSIFIER: Fake vs Real ===')
print(f'Accuracy  : {accuracy_score(y_test, y_pred_bin):.4f}')
print(f'F1 (macro): {f1_score(y_test, y_pred_bin, average="macro"):.4f}')
print()
print(classification_report(y_test, y_pred_bin, target_names=['Real','Fake']))


In [ ]:
cm = confusion_matrix(y_test, y_pred_bin)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Real','Fake'],
            yticklabels=['Real','Fake'])
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix — Binary Classifier')
plt.tight_layout(); plt.show()


In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob_bin)
roc_auc     = auc(fpr, tpr)
precision, recall, _ = precision_recall_curve(y_test, y_prob_bin)
avg_prec    = average_precision_score(y_test, y_prob_bin)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# ROC curve
ax1.plot(fpr, tpr, color='darkorange', lw=2,
         label=f'ROC curve (AUC = {roc_auc:.3f})')
ax1.plot([0,1],[0,1],'k--', lw=1)
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curve — Fake News Detection')
ax1.legend(loc='lower right')

# Precision-Recall curve
ax2.plot(recall, precision, color='steelblue', lw=2,
         label=f'PR curve (AP = {avg_prec:.3f})')
ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curve — Fake News Detection')
ax2.legend(loc='upper right')

plt.tight_layout(); plt.show()


### 6B — Multi-Class Category Classifier

Topic prototype vectors are constructed from GloVe keyword centroids. Each article is classified by cosine similarity to the six category prototypes.


In [ ]:
TOPIC_KEYWORDS = {
    'Politics':      ['government','election','president','congress','senate',
                       'law','vote','policy','minister','parliament','democrat'],
    'Technology':    ['technology','software','internet','computer','artificial',
                       'data','digital','innovation','startup','algorithm','cybersecurity'],
    'Sports':        ['game','player','team','score','championship','coach',
                       'match','tournament','league','athlete','stadium'],
    'Entertainment': ['movie','music','actor','celebrity','film','award',
                       'concert','television','Hollywood','streaming','director'],
    'Business':      ['market','stock','economy','investment','company',
                       'revenue','trade','finance','profit','merger','startup'],
    'Health':        ['health','disease','medical','hospital','vaccine',
                       'patient','doctor','treatment','pandemic','symptom','clinical'],
}

def build_prototype(keywords: list, glove: dict, dim: int = 100) -> np.ndarray:
    vecs = [glove[w] for w in keywords if w in glove]
    return np.mean(vecs, axis=0) if vecs else np.zeros(dim)

prototypes = {cat: build_prototype(kws, glove)
              for cat, kws in TOPIC_KEYWORDS.items()}
_proto_mat   = np.vstack(list(prototypes.values()))
_proto_norms = np.linalg.norm(_proto_mat, axis=1, keepdims=True)
_proto_normed= _proto_mat / (_proto_norms + 1e-9)
CATEGORIES   = list(prototypes.keys())

def predict_category(glove_vec: np.ndarray) -> tuple:
    """Returns (top_category, scores_dict)."""
    v      = glove_vec / (np.linalg.norm(glove_vec) + 1e-9)
    sims   = _proto_normed @ v
    scores = {cat: float(sims[i]) for i, cat in enumerate(CATEGORIES)}
    return max(scores, key=scores.get), scores

test_df[['predicted_category','cat_scores']] = test_df['glove_vec'].apply(
    lambda v: pd.Series(predict_category(v))
)
print('Category distribution in test set:')
print(test_df['predicted_category'].value_counts().to_string())


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
counts = test_df['predicted_category'].value_counts()
sns.barplot(x=counts.index, y=counts.values, ax=ax, palette='muted')
ax.set_title('Predicted News Category Distribution (Test Set)')
ax.set_xlabel('Category'); ax.set_ylabel('Count')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}',
                (p.get_x() + p.get_width()/2., p.get_height()),
                ha='center', va='bottom', fontsize=10)
plt.tight_layout(); plt.show()


---
## Task 7 — Unified Output Report

Runs a single article through the complete NLP pipeline and displays all results in a structured, readable report — matching the system architecture design.


In [ ]:
def analyze_article(raw_text: str) -> None:
    """End-to-end pipeline: preprocess → embed → POS → HMM → parse → classify."""
    SEP  = '=' * 68
    sep2 = '─' * 68

    print(SEP)
    print('   INTELLIGENT NEWS ARTICLE ANALYSIS REPORT')
    print('   NLP Assignment PS-5 | BITS Pilani AIMLCZG530')
    print(SEP)
    print(f'\nArticle preview:\n  {raw_text[:250]}...\n')

    # ── 1. Preprocess ──────────────────────────────────────────
    proc   = master_preprocessor(raw_text)
    print(sep2)
    print('[1] TEXT PREPROCESSING')
    print(f'    Total tokens     : {len(proc["sequential_tokens"])}')
    print(f'    Semantic tokens  : {len(proc["semantic_tokens"])}')
    print(f'    Political terms  : {proc["political_terms"][:8]}')

    # ── 2. GloVe ───────────────────────────────────────────────
    g_vec = tokens_to_glove_vec(proc['semantic_tokens'], glove)
    oov   = sum(1 for t in proc['semantic_tokens'] if t not in glove)
    print(sep2)
    print('[2] VECTOR EMBEDDING (GloVe 100d)')
    print(f'    Vector norm      : {np.linalg.norm(g_vec):.4f}')
    print(f'    OOV tokens       : {oov}/{len(proc["semantic_tokens"])}')
    print(f'    Nearest words    : {nearest_neighbors("news")[:3]}')

    # ── 3. Neural LM ───────────────────────────────────────────
    perp   = article_perplexity(proc['sequential_tokens'], lm_model, word_to_idx)
    preds  = predict_next_words(raw_text, lm_model, word_to_idx, idx_to_word, top_k=3)
    print(sep2)
    print('[3] NEURAL LANGUAGE MODEL')
    print(f'    Perplexity       : {perp:.2f}')
    print(f'    Next-word top-3  : {preds}')

    # ── 4. POS Tagging ─────────────────────────────────────────
    pos_res = pos_tag_article(raw_text)
    print(sep2)
    print('[4] POS TAGGING & NER')
    print(f'    Keywords         : {pos_res["keywords"][:6]}')
    print(f'    Action verbs     : {pos_res["action_verbs"][:5]}')
    print(f'    Political terms  : {pos_res["pol_terms"][:5]}')
    ents = [(e["Entity"], e["Type"]) for e in pos_res["entities"][:5]]
    print(f'    Named entities   : {ents}')

    # ── 5. Parsing ─────────────────────────────────────────────
    parse_res = parse_article(raw_text)
    print(sep2)
    print('[5] SYNTACTIC PARSING')
    print(f'    Noun phrases     : {parse_res["noun_phrases"][:4]}')
    if parse_res['relationships']:
        r = parse_res['relationships'][0]
        print(f'    Top SVO triplet  : ({r["Subject"]} | {r["Verb"]} | {r["Object"]})')
    print(f'    Total SVOs       : {parse_res["n_svos"]}')

    # ── 6. HMM ─────────────────────────────────────────────────
    hmm_f = hmm_features(raw_text)
    print(sep2)
    print('[6] HMM SEQUENCE MODEL')
    print(f'    LL (fake model)  : {hmm_f[0]:.2f}')
    print(f'    LL (real model)  : {hmm_f[1]:.2f}')
    print(f'    LL difference    : {hmm_f[2]:.2f}')

    # ── 7. Classification ──────────────────────────────────────
    pos_fv   = pos_feature_vector(pos_res['pos_counts'])
    parse_fv = parse_feature_vector(raw_text, len(proc['sequential_tokens']))
    feat = np.hstack([
        g_vec, hmm_f,
        np.array([min(perp,5000)/5000]),
        pos_fv, parse_fv
    ]).reshape(1, -1)
    pred_label = clf_binary.predict(feat)[0]
    pred_prob  = clf_binary.predict_proba(feat)[0]
    category, cat_scores = predict_category(g_vec)

    print(SEP)
    print('   FINAL VERDICT')
    print(SEP)
    veracity = 'FAKE NEWS' if pred_label == 1 else 'REAL NEWS'
    print(f'   📰  Predicted Category  : {category}')
    print(f'   🔍  Veracity Detection  : {veracity} '
          f'(confidence: {max(pred_prob):.1%})')
    print(f'   🏷️  Combined Label      : {veracity} — {category.upper()}')
    print()
    print('   Category similarity scores:')
    for cat, score in sorted(cat_scores.items(), key=lambda x: -x[1]):
        bar = '█' * int(score * 30)
        print(f'     {cat:<15s} {score:.4f}  {bar}')
    print(SEP)

# Run on first test article
analyze_article(test_df['full_content'].iloc[0])


In [ ]:
# ═══════════════════════════════════════════════════
# PERFORMANCE METRICS SUMMARY
# ═══════════════════════════════════════════════════
print('=' * 55)
print('  PERFORMANCE METRICS SUMMARY')
print('=' * 55)
print(f'  Test articles      : {len(test_df):,}')
print(f'  Binary accuracy    : {accuracy_score(y_test, y_pred_bin):.4f}')
print(f'  Binary F1 (macro)  : {f1_score(y_test, y_pred_bin, average="macro"):.4f}')
print(f'  ROC-AUC            : {roc_auc:.4f}')
print(f'  Avg Precision (PR) : {avg_prec:.4f}')
print()
print('  Binary classification report:')
print(classification_report(y_test, y_pred_bin,
      target_names=['Real','Fake'], digits=4))
print('  Predicted category distribution:')
for cat, cnt in test_df['predicted_category'].value_counts().items():
    print(f'    {cat:<15s}: {cnt:,}')
print('=' * 55)
